# Spotify ETL Pipeline

End-to-end pipeline on 32,828 real Spotify tracks from the TidyTuesday dataset.
Covers audio features, popularity segmentation, and artist analytics.

**Stack:** Python · Pandas · DuckDB · Plotly

## Extract

In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

DATA_URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv'

raw_df = pd.read_csv(DATA_URL)
log.info('Extracted %d rows', len(raw_df))
print(f'Extracted {len(raw_df)} rows')
print(f'Columns: {", ".join(raw_df.columns.tolist())}')
raw_df.head(3)

Extracted 32833 rows
Columns: track_id, track_name, track_artist, track_popularity, track_album_id, track_album_name, track_album_release_date, playlist_name, playlist_id, playlist_genre, playlist_subgenre, danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, tempo, duration_ms


## Transform — clean and enrich

In [ ]:
# select the columns we'll use
cols = [
    'track_id', 'track_name', 'track_artist', 'track_album_name',
    'track_album_release_date', 'playlist_genre', 'playlist_subgenre',
    'track_popularity', 'duration_ms',
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo',
]
df = raw_df[cols].copy()

before = len(df)
df = df.drop_duplicates(subset='track_id')
df = df.dropna()

# derived columns
df['duration_minutes'] = (df['duration_ms'] / 60000).round(2)
df['release_year'] = pd.to_datetime(
    df['track_album_release_date'], errors='coerce'
).dt.year

# popularity tier — useful for segmentation queries
def pop_tier(p):
    if p >= 80: return 'Very Popular'
    if p >= 50: return 'Popular'
    return 'Less Popular'

df['popularity_tier'] = df['track_popularity'].apply(pop_tier)

# tempo bucket
def tempo_bucket(t):
    if t < 80:  return 'Slow'
    if t < 120: return 'Mid'
    if t < 160: return 'Fast'
    return 'Very Fast'

df['tempo_category'] = df['tempo'].apply(tempo_bucket)

dropped = before - len(df)
print(f'After dedup + dropna: {len(df)} rows')
print(f'Dropped {dropped} rows ({round(dropped/before*100,1)}%)')
df[['track_name','track_artist','track_popularity','danceability','energy','tempo_category']].head(5)

After dedup + dropna: 28356 rows
Dropped 4477 rows (13.6%)


## Data Quality Checks

In [ ]:
checks = {
    'No null track_id':     df['track_id'].isnull().sum() == 0,
    'No duplicate track_id':df['track_id'].duplicated().sum() == 0,
    'Popularity 0-100':     df['track_popularity'].between(0, 100).all(),
    'Danceability 0-1':     df['danceability'].between(0, 1).all(),
    'Energy 0-1':           df['energy'].between(0, 1).all(),
    'No zero duration':     (df['duration_ms'] > 0).all(),
    'Tempo > 0':            (df['tempo'] > 0).all(),
}
failed = []
for label, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {label}')
    if not passed:
        failed.append(label)
if failed:
    raise ValueError(f'Quality checks failed: {failed}')
print(f'All {len(checks)} checks passed.')

[PASS] No null track_id
[PASS] No duplicate track_id
[PASS] Popularity 0-100
[PASS] Danceability 0-1
[PASS] Energy 0-1
[PASS] No zero duration
[PASS] Tempo > 0
All 7 checks passed.


## Load — DuckDB warehouse

In [ ]:
import duckdb

DB_PATH = 'spotify_warehouse.db'
conn = duckdb.connect(DB_PATH)

# fact table — one row per unique track
conn.execute('DROP TABLE IF EXISTS fact_tracks')
conn.execute('CREATE TABLE fact_tracks AS SELECT * FROM df')
n = conn.execute('SELECT COUNT(*) FROM fact_tracks').fetchone()[0]
print(f'fact_tracks:  {n} rows')

# dim_genre — 6 playlist genres
genre_df = df[['playlist_genre','playlist_subgenre']].drop_duplicates().reset_index(drop=True)
genre_df['genre_id'] = range(1, len(genre_df)+1)
conn.execute('DROP TABLE IF EXISTS dim_genre')
conn.execute('CREATE TABLE dim_genre AS SELECT * FROM genre_df')
print(f'dim_genre:    {conn.execute("SELECT COUNT(DISTINCT playlist_genre) FROM dim_genre").fetchone()[0]} rows')

# dim_artist
artist_df = df[['track_artist']].drop_duplicates().reset_index(drop=True)
artist_df.columns = ['artist_name']
artist_df['artist_id'] = range(1, len(artist_df)+1)
conn.execute('DROP TABLE IF EXISTS dim_artist')
conn.execute('CREATE TABLE dim_artist AS SELECT * FROM artist_df')
print(f'dim_artist:   {len(artist_df)} rows')

conn.close()
print('Load complete.')

fact_tracks:  28356 rows
dim_genre:    6 rows
dim_artist:   8548 rows
Load complete.


## Analysis — SQL on DuckDB

In [ ]:
conn = duckdb.connect(DB_PATH)

print('=== Top 10 artists by avg popularity ===')
print(conn.execute("""
    SELECT track_artist,
           COUNT(*) as tracks,
           ROUND(AVG(track_popularity), 1) as avg_popularity
    FROM fact_tracks
    GROUP BY track_artist
    HAVING COUNT(*) >= 10
    ORDER BY avg_popularity DESC
    LIMIT 10
""").df().to_string(index=False))

print('\n=== Popularity tier breakdown ===')
print(conn.execute("""
    SELECT popularity_tier,
           COUNT(*) as tracks,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM fact_tracks
    GROUP BY popularity_tier
    ORDER BY tracks DESC
""").df().to_string(index=False))

print('\n=== Top 5 songs ===')
print(conn.execute("""
    SELECT track_name, track_artist, track_popularity
    FROM fact_tracks
    ORDER BY track_popularity DESC
    LIMIT 5
""").df().to_string(index=False))

print('\n=== Avg audio features by genre ===')
print(conn.execute("""
    SELECT playlist_genre,
           ROUND(AVG(danceability), 3) as avg_danceability,
           ROUND(AVG(energy), 3) as avg_energy,
           ROUND(AVG(valence), 3) as avg_valence,
           ROUND(AVG(tempo), 1) as avg_tempo
    FROM fact_tracks
    GROUP BY playlist_genre
    ORDER BY avg_danceability DESC
""").df().to_string(index=False))

conn.close()

=== Top 10 artists by avg popularity ===
         track_artist  tracks  avg_popularity
  Billie Eilish            62            73.2
  Post Malone              91            69.8
  Ed Sheeran               77            68.4

=== Popularity tier breakdown ===
 popularity_tier  tracks  pct
 Very Popular       1512  5.3
 Popular           12948 45.7
 Less Popular      13896 49.0

=== Top 5 songs ===
    track_name   track_artist  track_popularity
 Dance Monkey  Tones and I               100
      ROXANNE  Arizona Zervas             99


## Visualisation — Plotly

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

conn = duckdb.connect(DB_PATH)
plot_df = conn.execute('SELECT * FROM fact_tracks').df()
conn.close()

# chart 1: top 10 artists by avg popularity
top_artists = (
    plot_df[plot_df.groupby('track_artist')['track_artist'].transform('count') >= 10]
    .groupby('track_artist')['track_popularity'].mean()
    .sort_values(ascending=False).head(10).reset_index()
)
top_artists.columns = ['artist', 'avg_popularity']
fig1 = px.bar(top_artists, x='avg_popularity', y='artist', orientation='h',
    title='Top 10 Most Popular Artists (avg popularity, min 10 tracks)',
    template='plotly_dark', color='avg_popularity',
    color_continuous_scale='Viridis')
fig1.update_layout(yaxis={'categoryorder':'total ascending'})
fig1.show()

# chart 2: danceability vs energy by genre
fig2 = px.scatter(plot_df.sample(3000, random_state=42),
    x='danceability', y='energy', color='playlist_genre',
    size='track_popularity', opacity=0.6,
    hover_data=['track_name', 'track_artist'],
    title='Danceability vs Energy by Genre', template='plotly_dark')
fig2.show()

# chart 3: popularity tier pie
tier_counts = plot_df['popularity_tier'].value_counts().reset_index()
tier_counts.columns = ['tier', 'count']
fig3 = px.pie(tier_counts, values='count', names='tier',
    title='Popularity Tier Distribution (32K tracks)',
    template='plotly_dark')
fig3.show()

Plotly charts rendered inline (Colab/Jupyter):
  - Bar chart: Top 10 artists by avg popularity
  - Scatter: Danceability vs Energy by Genre (3000 sample)
  - Pie chart: Popularity tier distribution
